# 🏠 Caso California Housing — Selección de variables con VIF y p-values

## 🎯 Objetivo
Construir un modelo de **regresión lineal múltiple** que prediga el **valor mediano de la vivienda** en distritos de California y decidir, con criterios estadísticos, **qué variables conservar**.

## 🧭 Lo que vas a aprender
1. Detectar **multicolinealidad** con el **VIF** (Variance Inflation Factor)
2. Detectar variables **no significativas** con el **p-value**
3. Eliminar variables **una a la vez** y comparar modelos con datos de prueba
4. Interpretar los coeficientes del modelo final

## 📋 El dataset
Cada fila es un **distrito censal** de California (censo de 1990).

| Variable | Descripción |
|---|---|
| `MedInc` | Ingreso mediano del distrito (en decenas de miles de USD: `3.5` = $35,000) |
| `HouseAge` | Edad mediana de las casas (años) |
| `AveRooms` | Promedio de cuartos por vivienda |
| `AveBedrms` | Promedio de recámaras por vivienda |
| `Population` | Población del distrito |
| `AveOccup` | Promedio de personas por vivienda |
| `Latitude` / `Longitude` | Ubicación del distrito |
| **`MedHouseVal`** | 🎯 **Variable objetivo:** valor mediano de la vivienda (en cientos de miles de USD: `2.5` = $250,000) |

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_frequencies(df, columnas, top_n=None)` | Frecuencia de variables categóricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_simple_regression(x, y, results)` | Recta ajustada de un modelo OLS con 1 variable |
| `plot_actual_vs_predicted(y_real, y_pred)` | Valores reales vs predichos |
| `plot_residuals(y_real, y_pred)` | Residuales vs predichos |
| `plot_rfecv(rfecv)` | R² según el número de variables seleccionadas por RFECV |

In [ ]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas',
        labels={col: col.capitalize() for col in columns}
    )
    fig.update_layout(width=1200, height=1200, title_font_size=20)
    fig.update_traces(diagonal_visible=True)
    fig.show()


def plot_simple_regression(x, y, results):
    """Dispersión de una variable vs el objetivo con la recta ajustada por un OLS de 1 variable."""
    b0, b1 = results.params.iloc[0], results.params.iloc[1]
    x_name = getattr(x, 'name', None) or 'x'
    y_name = getattr(y, 'name', None) or 'y'
    x_line = np.linspace(np.min(x), np.max(x), 100)

    fig = px.scatter(
        x=np.asarray(x),
        y=np.asarray(y),
        opacity=0.6,
        labels={'x': x_name, 'y': y_name},
        title=f'{y_name} = {b0:.2f} + ({b1:.4f}) · {x_name}',
        template='plotly_white'
    )
    fig.add_trace(go.Scatter(
        x=x_line,
        y=b0 + b1 * x_line,
        mode='lines',
        name='Recta OLS',
        line=dict(color='red', width=3)
    ))
    fig.show()


def plot_actual_vs_predicted(y_true, y_pred, title='Real vs Predicho'):
    """Valores reales vs predichos; un modelo perfecto cae sobre la diagonal roja."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())

    fig = px.scatter(
        x=y_true,
        y=y_pred,
        opacity=0.5,
        labels={'x': 'Valor real', 'y': 'Valor predicho'},
        title=title,
        template='plotly_white'
    )
    fig.add_shape(
        type='line', x0=lo, y0=lo, x1=hi, y1=hi,
        line=dict(color='red', dash='dash')
    )
    fig.show()


def plot_residuals(y_true, y_pred, title='Residuales vs Predicho'):
    """Residuales vs predichos; buscamos una nube sin patrón alrededor de 0."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)

    fig = px.scatter(
        x=y_pred,
        y=y_true - y_pred,
        opacity=0.5,
        labels={'x': 'Valor predicho', 'y': 'Residual (real − predicho)'},
        title=title,
        template='plotly_white'
    )
    fig.add_hline(y=0, line_dash='dash', line_color='red')
    fig.show()


def plot_rfecv(rfecv):
    """R² promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name='R² promedio (CV)'
    ))
    fig.update_layout(
        title='RFECV — R² según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title='R² (validación cruzada)',
        template='plotly_white',
        width=900, height=450
    )
    fig.show()

### 🧰 Funciones del caso

Estas funciones te ahorran código repetido; ejecuta la celda una vez:

| Función | ¿Para qué sirve? |
|---|---|
| `ajustar_ols(X_train, y_train)` | Ajusta un modelo OLS (ya agrega la constante) |
| `predecir(results, X)` | Genera predicciones con el modelo |
| `evaluar_modelo(nombre, results, X_test, y_test)` | R² y RMSE en el conjunto de prueba |
| `calcular_vif(X)` | VIF de cada variable, de mayor a menor |
| `plot_mapa_precios(df)` | Mapa de los distritos coloreado por precio |

In [ ]:
# Funciones del caso: ajustar, evaluar y diagnosticar modelos OLS
import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor


def ajustar_ols(X_train, y_train):
    """Ajusta un modelo OLS (agrega la constante automáticamente)."""
    return sm.OLS(y_train, sm.add_constant(X_train)).fit()


def predecir(results, X):
    """Predice con un modelo OLS ajustado con ajustar_ols()."""
    return results.predict(sm.add_constant(X))


def evaluar_modelo(nombre, results, X_test, y_test):
    """Imprime y regresa las métricas del modelo en el conjunto de prueba."""
    y_pred = predecir(results, X_test)
    metricas = {
        'modelo': nombre,
        'n_variables': X_test.shape[1],
        'R² ajustado (train)': round(results.rsquared_adj, 4),
        'R² (test)': round(r2_score(y_test, y_pred), 4),
        'RMSE (test)': round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
    }
    print(f"{nombre}: R² test = {metricas['R² (test)']} | RMSE test = {metricas['RMSE (test)']}")
    return metricas


def calcular_vif(X):
    """VIF de cada variable, de mayor a menor (se calcula con constante, igual que el modelo)."""
    X_const = sm.add_constant(X)
    vif = pd.DataFrame({
        'variable': X.columns,
        'VIF': [variance_inflation_factor(X_const.values, i + 1) for i in range(X.shape[1])],
    })
    return vif.sort_values('VIF', ascending=False).round(2).reset_index(drop=True)


def plot_mapa_precios(df):
    """Ubicación de cada distrito coloreada por el valor mediano de la vivienda."""
    fig = px.scatter(
        df,
        x='Longitude',
        y='Latitude',
        color='MedHouseVal',
        color_continuous_scale='Viridis',
        opacity=0.5,
        title='Valor mediano de la vivienda por ubicación',
        labels={'MedHouseVal': 'Valor (x $100k)'},
        template='plotly_white'
    )
    fig.update_yaxes(scaleanchor='x', scaleratio=1)
    fig.update_layout(width=750, height=700)
    fig.show()

## 1️⃣ Cargar los datos

El dataset viene incluido en scikit-learn; la primera vez se descarga automáticamente.

In [ ]:
from sklearn.datasets import fetch_california_housing

df = fetch_california_housing(as_frame=True).frame
df.head()

## 2️⃣ Conocer los datos

Antes de modelar respondemos tres preguntas: ¿cuántos datos hay?, ¿hay nulos?, ¿qué rangos tienen las variables?

In [ ]:
# ¿Cuántas filas y columnas hay? ¿Hay nulos?
# Pista: df.shape y df.isnull().sum()
# ✍️ Tu código aquí


In [ ]:
# Estadísticas descriptivas
# Pista: df.describe()
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Hay nulos? ¿Ves valores máximos extraños en alguna variable? ¿Cuál es el valor máximo de `MedHouseVal`?

RELLENAR CON TUS COMENTARIOS

## 3️⃣ Exploración visual

In [ ]:
# Distribución de la variable objetivo y del ingreso
# Llama a plot_distributions(df, ['MedHouseVal', 'MedInc'])
# ✍️ Tu código aquí


In [ ]:
# Matriz de correlación de todas las variables
# Llama a plot_correlation_matrix(df, df.columns)
# ✍️ Tu código aquí


In [ ]:
# Este va de regalo: ¿dónde están las casas más caras?
plot_mapa_precios(df)

✍️ **¿Qué observas?**
- ¿Qué variable está más correlacionada con `MedHouseVal`?
- ¿Qué pares de variables están muy correlacionadas **entre sí**?
- En el mapa, ¿dónde están las casas más caras?

RELLENAR CON TUS COMENTARIOS

## 4️⃣ Preparar los datos

Separamos la variable objetivo (`y`) de las predictoras (`X`) y dividimos **80% entrenamiento / 20% prueba**.
Todas las variables ya son numéricas: no necesitamos encoding.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. X = todas las columnas excepto 'MedHouseVal'; y = 'MedHouseVal'
# 2. train_test_split 80/20 con random_state=42
# ✍️ Tu código aquí


## 5️⃣ Modelo 1: todas las variables

Empezamos con el modelo completo como **punto de comparación**.

In [ ]:
# Ajusta el modelo con todas las variables y muestra el resumen
# Pista: modelo_1 = ajustar_ols(X_train, y_train)
#        print(modelo_1.summary())
# ✍️ Tu código aquí


In [ ]:
# Guardamos las métricas de cada modelo en esta lista para compararlos al final
comparacion = []

# Pista: comparacion.append(evaluar_modelo('1. Todas las variables', modelo_1, X_test, y_test))
# ✍️ Tu código aquí


## 6️⃣ Paso 1 — Multicolinealidad con VIF

El **VIF** mide qué tanto se puede explicar una variable **con las demás**. Si es alto, la variable es redundante y sus coeficientes se vuelven inestables.

| VIF | Interpretación |
|---|---|
| < 5 | ✅ Sin problema |
| 5 – 10 | ⚠️ Multicolinealidad moderada |
| > 10 | ❌ Multicolinealidad severa |

**Regla:** se elimina **una variable a la vez** y se vuelve a calcular el VIF, porque al quitar una, el VIF de las demás cambia.

In [ ]:
# Calcula el VIF de las variables de entrenamiento
# Pista: calcular_vif(X_train)
# ✍️ Tu código aquí


Hay dos **pares** con VIF alto. ¿Cuáles son?

### 🧪 Experimento A: quitar la variable con mayor VIF
La receta automática diría "quita la de mayor VIF". Veamos qué pasa:

In [ ]:
# 1. Crea cols_a = X_train.columns.drop('<variable con mayor VIF>')
# 2. modelo_a = ajustar_ols(X_train[cols_a], y_train)
# 3. evaluar_modelo('A. Sin <variable>', modelo_a, X_test[cols_a], y_test)
# ✍️ Tu código aquí


✍️ ¿Qué pasó con el R² en test comparado con el modelo 1? ¿Por qué crees que pasó? (Pista: recuerda el mapa)

RELLENAR CON TUS COMENTARIOS

### 🧪 Experimento B: quitar `AveBedrms`
¿`AveBedrms` aporta información que no esté ya en `AveRooms`?

In [ ]:
# 1. Crea cols_b = X_train.columns.drop('AveBedrms')
# 2. Vuelve a calcular el VIF con calcular_vif(X_train[cols_b])
# ✍️ Tu código aquí


In [ ]:
# Ajusta modelo_2 con cols_b, evalúalo y agrégalo a la lista comparacion
# ✍️ Tu código aquí


✍️ ¿Qué pasó con el VIF de `AveRooms`? ¿Y con el R² en test?

RELLENAR CON TUS COMENTARIOS

## 7️⃣ Paso 2 — Significancia con p-values

El **p-value** de cada coeficiente responde: *"si esta variable en realidad no tuviera efecto, ¿qué tan probable sería observar este coeficiente?"*

- **p < 0.05** → la variable es **significativa**: la conservamos
- **p ≥ 0.05** → no hay evidencia de que aporte: **candidata a eliminar**

In [ ]:
# Muestra los p-values del modelo_2, de mayor a menor
# Pista: modelo_2.pvalues.round(4).sort_values(ascending=False)
# ✍️ Tu código aquí


✍️ ¿Qué variable tiene p-value mayor a 0.05? Elimínala en la siguiente celda.

In [ ]:
# 1. cols_final = cols_b.drop('<variable no significativa>')
# 2. Ajusta modelo_final, evalúalo y agrégalo a comparacion
# 3. Revisa que todos los p-values sean < 0.05
# ✍️ Tu código aquí


## 8️⃣ Comparar modelos

In [ ]:
# Muestra la tabla comparativa
# Pista: pd.DataFrame(comparacion)
# ✍️ Tu código aquí


✍️ ¿Qué modelo elegirías y por qué?

RELLENAR CON TUS COMENTARIOS

## 9️⃣ Diagnóstico del modelo final

In [ ]:
# 1. y_pred_final = predecir(modelo_final, X_test[cols_final])
# 2. Llama a plot_actual_vs_predicted(y_test, y_pred_final) y a plot_residuals(y_test, y_pred_final)
# ✍️ Tu código aquí


✍️ ¿Ves alguna línea o patrón extraño en las gráficas? (Pista: recuerda el tope de `MedHouseVal`)

RELLENAR CON TUS COMENTARIOS

## 🔟 Interpretar los coeficientes

In [ ]:
# Muestra los coeficientes del modelo final
# Pista: modelo_final.params.round(4)
# ✍️ Tu código aquí


✍️ Interpreta el coeficiente de `MedInc` en palabras. Recuerda las unidades: `MedInc` está en decenas de miles de USD y `MedHouseVal` en cientos de miles de USD.

RELLENAR CON TUS COMENTARIOS

## 📝 Conclusiones
RELLENAR: ¿qué aprendiste sobre el VIF y los p-values?